# 训练后端切换原理

这次切换保留现有的 Wordle RL 流水线，只替换 Actor 与 Ref 的执行引擎。开始修改配置前，先看清三件事：哪些训练逻辑继续沿用，哪些配置交给新后端，以及更新后的 Actor 权重怎样传回 vLLM。


<img src="./images/backend_boundary.png" alt="训练后端切换边界" width="90%">

## 保持不变与发生变化的部分

| 类型 | 组件 | 说明 |
| --- | --- | --- |
| 保持不变 | Qwen3 Wordle 模型、训练数据、验证数据 | 复用前四章资产 |
| 保持不变 | GRPO | 仍按组内奖励计算相对优势 |
| 保持不变 | vLLM rollout | 仍负责异步生成与采样 |
| 保持不变 | Wordle AgentLoop 与奖励函数 | 六轮交互和评分语义不变 |
| 发生变化 | Actor | 参数分片、前反向、优化器更新和内存管理改由 TorchTitan Engine 执行 |
| 发生变化 | Ref | log probability 前向计算改由 TorchTitan Engine 执行 |

可见，选择新的 engine 只是第一步。Actor、Ref 和权重同步仍要沿用 Verl 约定的接口，整条 RL 流水线才能继续工作。


## 配置映射

| 配置职责 | 原 FSDP 路径 | TorchTitan-NPU 路径 | 处理方式 |
| --- | --- | --- | --- |
| 选择训练引擎 | 使用 Verl 默认 FSDP engine | `model_engine=torchtitan` | 新增决定性选择器 |
| Actor 后端配置 | `actor.fsdp_config.*` | `actor.torchtitan.*` | 切换配置命名空间 |
| Ref 后端配置 | Ref 的 FSDP 配置 | `ref.torchtitan.*` | 与 Actor 使用同类引擎接口 |
| 数据并行分片 | FSDP world size | `data_parallel_shard_size=2` | 映射到两卡 FSDP2 基线的 shard mesh |
| 参数与优化器卸载 | FSDP offload 配置 | `param_offload`、`optimizer_offload` | 保留卸载目标，改由 TorchTitan 管理 |
| rollout | `rollout.name=vllm` | `rollout.name=vllm` | 保持不变 |
| 训练算法 | `adv_estimator=grpo` | `adv_estimator=grpo` | 保持不变 |

`actor.fsdp_config` 和 `actor.torchtitan` 分别由两个 engine 读取，切换后应使用对应的配置域。后续章节采用两卡 FSDP2，并为训练计算启用 TorchTitan-NPU 的 TND 变长注意力。上下文并行留到长序列扩展部分介绍，三步训练仍使用 CP 1。


<img src="./images/rl_backend_dataflow.png" alt="训练后端数据流和权重同步" width="90%">

## 从 Verl 到 TorchTitan Engine 的调用链

1. Hydra 解析启动参数，`model_engine=torchtitan` 进入 Verl 训练配置。
2. Verl unified worker 根据后端选择器，为 Actor 与 Ref 创建 TorchTitan Engine。
3. TorchTitan Engine 根据模型配置初始化 Qwen3，并构建 DeviceMesh 与 FSDP2 分片。
4. vLLM 完成 rollout 后，Wordle AgentLoop 和奖励函数得到完整轨迹及奖励。
5. Actor 与 Ref 的 TorchTitan Engine 分别计算 old log-prob 与 ref log-prob。
6. Verl 根据奖励与 log-prob 计算 GRPO 优势，再将训练 batch 交给 Actor engine；TorchTitan 执行 Actor 前向、反向和优化器更新。
7. Verl 调用 engine 的 `get_per_tensor_param()` 读取最新 Actor 状态。TorchTitan 的 `StateDictAdapter` 将参数名从 TorchTitan 格式转换为 vLLM 所需的 Hugging Face 格式；若参数是 DTensor，同步路径还会将分片物化为完整 tensor。
8. unified worker 调用 rollout 的 `update_weights()` 将最新 Actor 权重交给 vLLM，下一轮 rollout 因而使用更新后的策略。

这里的权重同步发生在每轮 RL 训练内部：Actor 更新后，vLLM 随即拿到新策略。checkpoint 则用于保存训练状态，供后续任务恢复，两者处在不同的环节。


## 课后练习

### 判断题

1. （判断题）切换训练后端后，Wordle 奖励函数和 GRPO 优势计算应保持不变。

2. （判断题）Actor 参数以 DTensor 分片保存时，权重同步链路需要在导出阶段正确物化完整 tensor。

### 单选题

3. （单选题）决定 Actor 与 Ref 使用 TorchTitan 训练引擎的关键配置是哪一项？

   A. `model_engine=torchtitan`

   B. `actor_rollout_ref.rollout.name=vllm`

   C. `algorithm.adv_estimator=grpo`

   D. `trainer.device=npu`

### 多选题

4. （多选题）下列哪些内容位于训练后端切换边界内？

   A. Actor 的参数分片与反向传播

   B. Actor 的优化器状态管理

   C. Reference Model 的分片前向计算

   D. Wordle 的奖励规则

5. （多选题）Actor 完成参数更新后，权重同步到 vLLM 的作用包括哪些？

   A. 让下一轮 rollout 使用最新策略

   B. 连接训练阶段与生成阶段

   C. 替代 GRPO 的优势计算

   D. 避免 vLLM 长期使用旧权重


> 完成练习后，运行下方单元格查看参考答案和解析。


In [ ]:
from pathlib import Path
import subprocess

course_root = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip())
answer_path = course_root / 'tutorials/rl_training_pipeline/05_training_backend/answer/05.02_answer.txt'
assert answer_path.is_file(), f'未找到答案文件: {answer_path}'
print(answer_path.read_text(encoding='utf-8'))
